# Drive and utilities


In [15]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [16]:
#@title Utils
!pip install dill
!pip install pandas
!pip install numpy
!
import dill as pickle



In [17]:
#@title Clone the repo and add API key
!git clone https://github.com/amusali/bert_for_patents.git
import os
# Api key setting
os.environ['patentsview_api_key'] = 'Uch9R1RR.BxaQcxV8HK4nBMgTTjX1F9CQW8PF4nXw'  # Replace with your actual key

Cloning into 'bert_for_patents'...
remote: Enumerating objects: 1983, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 1983 (delta 148), reused 211 (delta 133), pack-reused 1742 (from 2)
Receiving objects: 100% (1983/1983), 206.70 MiB | 23.74 MiB/s, done.
Resolving deltas: 100% (1107/1107), done.


In [18]:
#@title Run setup file to get environment
#!python "/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py"

import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

In [19]:
#@title Pull latest changes from repo
%cd "/content/bert_for_patents"
!git pull origin master
#@title Add paths to the system and change directory
import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents
From https://github.com/amusali/bert_for_patents
 * branch            master     -> FETCH_HEAD
Already up to date.


# Save files into Colab

In [7]:
# Example: copy MyDrive/data_big to local /content/data_big
!mkdir -p "/content/matches"
!rsync -a --info=progress2 \
  --include='01 Hybrid matches (lambda 0.6 and 0.7) - *10matches.pkl' \
  --exclude='*' \
  "/content/drive/MyDrive/PhD Data/11 Matches/actual results/paper/" \
  "/content/matches/"


    479,867,144 100%   21.79MB/s    0:00:20 (xfr#24, to-chk=0/25)


In [8]:
# --- Imports
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

# ---------- CONFIG ----------
# Folder that contains your per-file outputs (with treated_id / control_id)
DATA_DIR = Path("/content/matches/")

# Metadata with treated patent info: patent_id, grant_date (or grant_q), acq_date
# Accepts .csv / .parquet / .dta
META_PATH = Path("/content/drive/MyDrive/PhD Data/09 Acquired patents/04 All patents.dta")

# Column names in the metadata file
META_ID_COL      = "patent_id"
META_GRANT_DATE  = "grant_date"       # date-like or YYYY-MM-DD; or give 'grant_q' as 'YYYYQn'
META_ACQ_DATE    = "acq_date"         # date-like or YYYY-MM-DD

# ---------- HELPERS ----------
def smart_read(path: Path) -> pd.DataFrame:
    """Read csv/parquet/dta/pkl with pandas if available."""
    ext = path.suffix.lower()
    if ext == ".csv":
        return pd.read_csv(path)
    if ext in (".parquet", ".pq"):
        return pd.read_parquet(path)
    if ext == ".pkl":
        return pd.read_pickle(path)
    if ext == ".dta":
        return pd.read_stata(path)

    raise ValueError(f"Unsupported file type: {path}")

def to_quarter_from_date(s):
    """Convert a date-like Series to pandas Period(Q)."""
    s = pd.to_datetime(s, errors="coerce")
    return s.dt.to_period("Q")

def to_quarter_from_str_q(s):
    """Handles strings like '2014Q3' -> Period('2014Q3')."""
    s = s.astype(str).str.upper().str.replace(" ", "")
    # Allow 'YYYYQn' or 'YYYY-Qn'
    s = s.str.replace("-", "", regex=False)
    return pd.PeriodIndex([q if "Q" in q else np.nan for q in s], freq="Q")

def ensure_quarter(col, try_as_date=True):
    """Return a Period(Q) Series from date or 'YYYYQn' strings."""
    if try_as_date:
        q = to_quarter_from_date(col)
        if q.notna().any():
            return q
    # fall back to parsing 'YYYYQn'
    return to_quarter_from_str_q(col)


def parse_suffix(suffix: str) -> dict:
    parts = [p.strip() for p in suffix.split(',')]
    params = { 'acq_type': parts[0] }
    if parts[1].lower() == 'baseline':
        params.update({
            'top_tech': False,
            'top_tech_threshold': None,
            'baseline_period': int(parts[2].rstrip('q')),
            'caliper': float(parts[3].replace('caliper_', '')),
            'K': int(parts[4].replace('matches', ''))
        })
    else:
        params.update({
            'top_tech': True,
            'top_tech_threshold': int(parts[2]),
            'baseline_period': int(parts[3].rstrip('q')),
            'caliper': float(parts[4].replace('caliper_', '')),
            'K': int(parts[5].replace('matches', ''))
        })
    return params

In [9]:

# ---------- 1) STACK ids from all files ----------
long_rows = []
files = sorted([p for p in DATA_DIR.iterdir() if p.suffix.lower() in (".csv",".parquet",".pq",".dta",".pkl")])
assert len(files) == 2 * 2 * 6, f"Expected 120 files, but found {len(files)}"

# Map files for later lookup
file_map = pd.DataFrame({
    "file_id": range(1, len(files)+1),
    "filename": [p.name for p in files],
    "path": [str(p) for p in files]  # optional, if you want full paths
})
print(len(file_map))

if not files:
    raise FileNotFoundError(f"No data files found in {DATA_DIR}")

for fid, p in tqdm(zip(file_map["file_id"], files), desc = "Processing files"):
    # Parse suffix and get parameters
    #params = parse_suffix(p.stem.replace("01 Hybrid matches - ", ""))


    # Load Pickle file
    sample_dict = smart_read(p)

    # Collapse the dictionary into a DF
    df = pd.concat(
        [v.assign(lambda_val=round(k, 2)) for k, v in sample_dict.items()],
        ignore_index=True
    )

    # Filter matches with very small cosine distance (if lambda is not 0) or very high Mahalanabois distance (if lambda is not 1)
    df = df[(df["cosine_distance"] >= 0.01) & (df["mahalanobis_distance"] <= 10)]


    # Require these columns in each file:
    for req in ("lambda_val", "treated_id","control_id"):
        if req not in df.columns:
            raise KeyError(f"{p.name} missing required column: {req}")

    # ID check
    if df.duplicated(subset=["lambda_val", "treated_id", "control_id"]).any():
        raise ValueError("Lambda and IDs do NOT uniquely identify observations!")

    # Keep relevant columns only
    df = df[["lambda_val", "treated_id","control_id"]]

    # Long format: treated row + control row, with treated flag
    treated_part = df[["lambda_val", "treated_id"]].rename(columns={"treated_id":"id"})
    treated_part["treated"] = 1
    control_part = df[["lambda_val", "control_id"]].rename(columns={"control_id":"id"})
    control_part["treated"] = 0

    out = pd.concat([treated_part, control_part], ignore_index=True).drop_duplicates()
    out["file_id"] = fid
    long_rows.append(out)

ids_long = pd.concat(long_rows, ignore_index=True).drop_duplicates()


24


Processing files: 24it [00:12,  1.95it/s]


In [20]:
from pathlib import Path
output_dir = Path("/content/drive/MyDrive/PhD Data/12 Sample Final/actual results/paper")
output_dir.mkdir(parents=True, exist_ok=True)

# Save this data into drive as Pickle and DTA
ids_long.to_pickle(output_dir / "00 Matched IDs - no pair info.pkl")
ids_long.to_stata(output_dir / "00 Matched IDs - no pair info.dta", write_index=False)

# Save filemaps as DTA and CSV
file_map.to_stata(output_dir / "00 Matched IDs - file map.dta", write_index=False)
file_map.to_csv(output_dir / "00 Matched IDs - file map.csv", index=False)

In [13]:
ids_long.head()

,lambda_val,id,treated,file_id
0,0.6,5241777,1,1
1,0.6,6404426,1,1
2,0.6,7212202,1,1
3,0.6,8189000,1,1
4,0.6,9877531,1,1


In [ ]:
sample_dict = smart_read(p)

# Collapse the dictionary into a DF
df = pd.concat(
    [v.assign(lambda_val=round(k, 2)) for k, v in sample_dict.items()],
    ignore_index=True
)

In [12]:
len(df[(df["cosine_distance"] >= 0.01) & (df["mahalanobis_distance"] <= 10)])


KeyError: 'cosine_distance'

In [11]:
len(df)

38452